In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
start_year = 2000
end_year = 2024
pagination_limit = 1000  # Limite de pagination pour les requêtes


indicators = {
    "population": "SP_POP_TOTL",
    "land_area_km2": "AG_LND_TOTL_K2",
    "gdp_per_capita_ppp_current": "NY_GDP_PCAP_PP_CD",
    "gdp_per_capita_usd_current": "NY_GDP_PCAP_CD",
    "life_expectancy": "SP_DYN_LE00_IN",
    "co2_emissions_per_capita": "EN_GHG_CO2_PC_CE_AR5",
    "pm2_5_exposure": "EN_ATM_PM25_MC_M3",
    'urban_population_percentage': "SP_URB_TOTL_IN_ZS",
    'health_expenditure_percentage_of_gdp': "SH_XPD_CHEX_GD_ZS",
}

# Indicator 1 : Population

## 1. Test code

In [4]:
df_test = pd.DataFrame({'REF_AREA': ['FRA', 'FRA', 'FRA','FRA','ITA'],
                       'TIME_PERIOD': [2020, 2020, 2021, 2021, 2021],
                       'OBS_VALUE': [100, 200, 200, np.nan, 400]})

# clé : pays * date, valeur : OBS_VALUE
# j'attends 3 groupes : (FRA, 2020), (FRA, 2021) et (ITA, 2021)
grouped = df_test.groupby(['REF_AREA', 'TIME_PERIOD']).size()
#print(grouped)

# 1. Filtrer la Series
filtre = grouped[grouped > 1]

print(f"Filtre : {filtre}")


# 2. Obtenir le nombre d'éléments
nombre_count = filtre.count()  # ou len(filtre)

# 3. Récupérer les clés (l'index)
cles = filtre.index.tolist()

#print(f"Clés correspondantes : {cles}")

Filtre : REF_AREA  TIME_PERIOD
FRA       2020           2
          2021           2
dtype: int64


## 2. Unicity of rows 

We check if there is 1 unique row for each key (year x area)

In [5]:
df_pop = pd.read_csv(Path(f"../data/raw/population_{start_year}-{end_year}_raw.csv"))
print("================================")
print(f"indicateur : ", "population")
print(f"Lecture du fichier pour l'indicateur population : ",df_pop.shape)

#on groupe selon la clé
pop_counts_per_key = df_pop.groupby(['TIME_PERIOD', 'REF_AREA']).size()

# 1. Filtrer la Series
df_pop_filtre = pop_counts_per_key[pop_counts_per_key > 1]

# 2. Obtenir le nombre d'éléments
df_pop_nombre = df_pop_filtre.count()  # ou len(filtre)

# 3. Récupérer les clés (l'index)
df_pop_cles = df_pop_filtre.index.tolist()

print(f"Nombre d'éléments > 1 : {df_pop_nombre}")
print(f"Clés correspondantes : {df_pop_cles}")


indicateur :  population
Lecture du fichier pour l'indicateur population :  (6600, 24)
Nombre d'éléments > 1 : 0
Clés correspondantes : []


We check if there any missing value in OBS_VALUE and if the numbre of keys is equal to the number of rows

In [6]:
# Combien d'années 
df_pop_annees = df_pop['TIME_PERIOD'].nunique()
print(f"Nombre d'années : {df_pop_annees}")

# Combien de pays
df_pop_pays = df_pop['REF_AREA'].nunique()
print(f"Nombre de pays : {df_pop_pays}")

nb_lignes = df_pop.shape[0]
nb_keys = df_pop['REF_AREA'].nunique() * df_pop['TIME_PERIOD'].nunique()
print(f"Nombre de clés : {nb_keys}")
print(f"Nombre de lignes : {nb_lignes}")

nb_missing_values = df_pop['OBS_VALUE'].isnull().sum()
print(f"Nombre de valeurs manquantes : {nb_missing_values}")


Nombre d'années : 25
Nombre de pays : 264
Nombre de clés : 6600
Nombre de lignes : 6600
Nombre de valeurs manquantes : 0


Everything seems to be all right.

We check the number of area : 264 is way big than the excpeted number of countries

In [7]:
df_pop.nunique()

OBS_VALUE           6544
TIME_FORMAT            1
UNIT_MULT              1
COMMENT_OBS            0
OBS_STATUS             1
OBS_CONF               1
AGG_METHOD             1
DECIMALS               1
COMMENT_TS             0
DATA_SOURCE            0
LATEST_DATA            1
DATABASE_ID            1
INDICATOR              1
REF_AREA             264
SEX                    1
AGE                    1
URBANISATION           1
COMP_BREAKDOWN_1       1
COMP_BREAKDOWN_2       1
COMP_BREAKDOWN_3       1
TIME_PERIOD           25
FREQ                   1
UNIT_MEASURE           1
UNIT_TYPE              0
dtype: int64

In [8]:
df_pop = pd.read_csv(Path(f"../data/raw/population_{start_year}-{end_year}_raw.csv"))


#we drop all columns with a unique value count of 1 or 0 except units
df_pop = df_pop[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE', 'UNIT_MULT', 'UNIT_MEASURE','UNIT_TYPE']]
print(df_pop) 

# we drop units if unnecessary
df_pop = df_pop[['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']]
print(df_pop)

     REF_AREA  TIME_PERIOD   OBS_VALUE  UNIT_MULT UNIT_MEASURE  UNIT_TYPE
0         YEM         2015    31159379          0           PS        NaN
1         ZMB         2015    16399089          0           PS        NaN
2         ZWE         2015    14399013          0           PS        NaN
3         AFE         2016   623369401          0           PS        NaN
4         AFW         2016   428797338          0           PS        NaN
...       ...          ...         ...        ...          ...        ...
6595      SSF         2015  1024659421          0           PS        NaN
6596      SSA         2015  1024566002          0           PS        NaN
6597      TSS         2015  1024659421          0           PS        NaN
6598      UMC         2015  2916662642          0           PS        NaN
6599      WLD         2015  7441084771          0           PS        NaN

[6600 rows x 6 columns]
     REF_AREA  TIME_PERIOD   OBS_VALUE
0         YEM         2015    31159379
1        

In [9]:
print(df_pop['REF_AREA'].unique())

<StringArray>
['YEM', 'ZMB', 'ZWE', 'AFE', 'AFW', 'ARB', 'CSS', 'CEB', 'EAR', 'EAS',
 ...
 'SLB', 'SOM', 'ZAF', 'SSD', 'DMA', 'SAS', 'TSA', 'SSF', 'DOM', 'ECU']
Length: 264, dtype: str


In [10]:
print(df_pop['REF_AREA'].nunique())

264


There are 264 areas which includes countries and **aggregates**

Let's drop aggregates and conserve countries only

In [ ]:
import requests

per_page = 400 # a huge number in order have everything in 1 call : if total < per_page


url = "https://api.worldbank.org/v2/country?"
response = requests.get(url,params={"format":"json", "per_page": per_page})
response.raise_for_status()

print(response.url)
print(response.text[:300])

area_data = response.json()


print("Nombre de id_pays",area_data[0]['total'])
print("id_pays_aruba",area_data[1][0]['id'])
print("id_pays_2",area_data[1][1]['id'])

print("nb pays",len(area_data[1]))

L_id_countries = []

for pays in range(len(area_data[1])):
    L_id_countries.append(area_data[1][pays]['id'])
print(L_id_countries)
print(len(L_id_countries))

https://api.worldbank.org/v2/country?format=json&per_page=400
[{"page":1,"pages":1,"per_page":"400","total":295},[{"id":"ABW","iso2Code":"AW","name":"Aruba","region":{"id":"LCN","iso2code":"ZJ","value":"Latin America & Caribbean "},"adminregion":{"id":"","iso2code":"","value":""},"incomeLevel":{"id":"HIC","iso2code":"XD","value":"High income"},"lendingType":{"
Nombre de id_pays 295
id_pays_aruba ABW
id_pays_2 AFE
nb pays 295
['ABW', 'AFE', 'AFG', 'AFR', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEA', 'BEC', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHI', 'BHR', 'BHS', 'BIH', 'BLA', 'BLR', 'BLZ', 'BMN', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BSS', 'BTN', 'BWA', 'CAA', 'CAF', 'CAN', 'CEA', 'CEB', 'CEU', 'CHE', 'CHI', 'CHL', 'CHN', 'CIV', 'CLA', 'CME', 'CMR', 'COD', 'COG', 'COL', 'COM', 'CPV', 'CRI', 'CSA', 'CSS', 'CUB', 'CUW', 'CYM', 'CYP', 'CZE', 'DEA', 'DEC', 'DEU', 'DJI', 'DLA', 'DMA', 'DMN', 'DNK', 'DNS', 'DOM', 'DSA', 'DSF', 'DSS', 'DZ

In [36]:
import pandas as pd

df_area = pd.json_normalize(area_data[1])
# Should have 295  rows
df_area.shape

(295, 18)

In [37]:
df_area.head()

,id,iso2Code,name,capitalCity,longitude,latitude,region.id,region.iso2code,region.value,adminregion.id,adminregion.iso2code,adminregion.value,incomeLevel.id,incomeLevel.iso2code,incomeLevel.value,lendingType.id,lendingType.iso2code,lendingType.value
0,ABW,AW,Aruba,Oranjestad,-70.0167,12.5167,LCN,ZJ,Latin America & Caribbean,,,,HIC,XD,High income,LNX,XX,Not classified
1,AFE,ZH,Africa Eastern and Southern,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates
2,AFG,AF,Afghanistan,Kabul,69.1761,34.5228,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakis...",LIC,XM,Low income,IDX,XI,IDA
3,AFR,A9,Africa,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates
4,AFW,ZI,Africa Western and Central,,,,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates


In [40]:
df_area['region.value'].value_counts()

df_area[df_area['region.value']!= 'Aggregates'].shape

(217, 18)